# DINOv3 ViT — unsupervised chinee apple localization

**Stage 1: model first, labels later.** No manual annotation. We run a frozen
DINOv3 ViT-B/16 backbone, cluster its per-patch features with KMeans, and you
assign *one integer* — "cluster N is chinee apple" — after looking at the
overlay. Those cluster ids are the pseudo-labels for the linear probe in stage 2.

```
tile -> DINOv3 patch tokens -> L2-norm -> KMeans(k) -> pick cluster -> mask + heat + boxes
```

Set **Runtime -> Change runtime type -> GPU (T4)** before running.

## 1. Setup — deps + this repo

In [ ]:
!pip install -q scikit-learn scipy matplotlib
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os, sys
REPO = '/content/weed-detection-experiments'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/nutboltu/weed-detection-experiments.git {REPO}
sys.path.insert(0, os.path.join(REPO, 'dinov3-vit'))
import vit_features as vf
print('vit_features loaded from', vf.__file__)

## 2. Load the backbone

**DINOv3 is gated.** Accept Meta's license, download the ViT-B/16 weights
(https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/), then in a
cell run `!git clone https://github.com/facebookresearch/dinov3.git /content/dinov3`,
upload the `.pth`, and set `FAMILY='dinov3'` with the two paths below.

**Until then, `FAMILY='dinov2'`** loads an ungated ViT-B/14 from `torch.hub` and
runs the identical pipeline — swap later with zero code changes.

In [ ]:
FAMILY = 'dinov2'            # 'dinov3' (gated) or 'dinov2' (ungated, works today)
REPO_DINOV3 = None          # e.g. '/content/dinov3'
WEIGHTS_DINOV3 = None       # e.g. '/content/dinov3_vitb16_pretrain_lvd1689m.pth'

if FAMILY == 'dinov3' and not (REPO_DINOV3 and WEIGHTS_DINOV3):
    raise SystemExit("DINOv3 is gated: set REPO_DINOV3 + WEIGHTS_DINOV3, or use FAMILY='dinov2'.")

model, meta = vf.load_backbone(FAMILY, repo=REPO_DINOV3, weights=WEIGHTS_DINOV3)
print('loaded', meta)

## 3. Upload a UAV tile

In [ ]:
from google.colab import files
up = files.upload()
IMAGE = next(iter(up))
print('using', IMAGE)

## 4. Extract features + cluster (the annotation-free step)

In [ ]:
import matplotlib.pyplot as plt
K = 6
feats, grid = vf.patch_features(model, IMAGE, meta, img_size=768)
labels, centroids = vf.cluster_patches(feats, k=K)
print('patches:', feats.shape, '| grid:', grid)

overlay = vf.cluster_overlay(IMAGE, labels, grid, K)
plt.figure(figsize=(9, 9)); plt.imshow(overlay); plt.axis('off')
plt.title(f'KMeans clusters (k={K})'); plt.show()

### 4b. Per-cluster panels — makes the pick obvious

In [ ]:
import numpy as np
from PIL import Image
base = np.asarray(Image.open(IMAGE).convert('RGB'))
W, H = Image.open(IMAGE).size
lbl_full = vf.upsample_grid(vf.cluster_grid(labels, grid), (W, H), nearest=True).astype(int)
fig, axs = plt.subplots(1, K, figsize=(3 * K, 3))
for c in range(K):
    m = (lbl_full == c)[..., None]
    axs[c].imshow((base * (0.2 + 0.8 * m)).astype('uint8'))
    axs[c].set_title(f'cluster {c}'); axs[c].axis('off')
plt.show()

## 5. Pick the chinee-apple cluster and localize

Set `TARGET_CLUSTER` to the id whose panel above best covers the weed. That one
integer is the only human input in the whole pipeline.

In [ ]:
TARGET_CLUSTER = 3   # <-- change to the chinee-apple cluster id from 4b
THRESHOLD = 0.5

result = vf.localize(feats, labels, centroids, TARGET_CLUSTER, grid, IMAGE,
                     heat_threshold=THRESHOLD, min_area=64)

from PIL import ImageDraw
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(result['heat'], cmap='inferno'); axs[0].set_title('P(chinee apple)'); axs[0].axis('off')
axs[1].imshow(result['mask'], cmap='gray'); axs[1].set_title('mask'); axs[1].axis('off')
boxed = result['base_image'].copy(); d = ImageDraw.Draw(boxed)
for (x0, y0, x1, y1, _a) in result['boxes']:
    d.rectangle([x0, y0, x1, y1], outline='lime', width=3)
axs[2].imshow(boxed); axs[2].set_title(f"{len(result['boxes'])} boxes"); axs[2].axis('off')
plt.show()

vf.save_outputs(result, '/content/out', stem='tile')

## 6. Next → stage 2 (linear probe)

When the picks look right, promote the cluster ids to pseudo-labels:

1. Loop `patch_features` + `cluster_patches` over several tiles, record
   `(feats, cluster_id)`.
2. "Chosen cluster" = positive, rest = negative → logistic regression / small
   MLP on the frozen 768-d tokens.
3. Apply the probe per patch on new tiles — no re-clustering, and it transfers
   across tiles where per-tile KMeans ids would not.

This is the "line probe" step you wanted to defer until the backbone works.